# DNN v2 Export — ContextConditionedTCN (FiLM + Temporal Conv)

Train the winning architecture from `01_architecture.ipynb` on 80% of data, calibrate on validation, export production artifacts.

**Architecture:** ContextConditionedTCN — FiLM-conditioned temporal convolution. Cross-candle indicators (22) modulate temporal processing of raw snapshot sequences (11).

**Features:** 33 total (11 raw + 22 cross-candle), temporal mode.

**Data:** `elapsed_pct <= 0.50` cutoff to prevent leakage.

In [1]:
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, "../..")
from polybot.adapters.dnn_models import ContextConditionedTCN

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_COLS = [
    "btc_price",
    "elapsed_pct",
    "market_volume",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
]
CROSS_CANDLE_COLS = [
    "prior_return",
    "consecutive_streak",
    "streak_magnitude",
    "rolling_volatility",
    "candle_momentum",
    "ma_crossover",
    "trend_consistency",
    "reversal_regime",
    "rsi",
    "bollinger_pct_b",
    "stochastic_k",
    "adx",
    "return_autocorrelation",
    "multi_candle_return_3",
    "multi_candle_return_6",
    "regime_score",
    "mean_reversion_signal",
    "prior_reversal_rate",
    "volume_momentum",
    "volume_trend",
    "volume_price_correlation",
    "relative_volume",
]
ALL_COLS = RAW_COLS + CROSS_CANDLE_COLS
ELAPSED_CUTOFF = 0.50
SEQ_LEN = 50
MODELS_DIR = "../../models"
DATA_DIR = "../../data"

## 1. Load Data & Build Temporal Sequences

In [2]:
rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
df[ALL_COLS] = df[ALL_COLS].fillna(0)

df_mid = df[df["elapsed_pct"] <= ELAPSED_CUTOFF].copy()
print(f"Snapshots: {len(df_mid):,} / {len(df):,} (cutoff={ELAPSED_CUTOFF})")

# Build temporal sequences
candle_ids = df_mid["candle_id"].unique()
sequences, targets = [], []
for cid in candle_ids:
    group = df_mid[df_mid["candle_id"] == cid]
    feats = group[ALL_COLS].values.astype(np.float32)
    target = int(group["target"].iloc[0])
    if len(feats) < SEQ_LEN:
        pad = np.tile(feats[0:1], (SEQ_LEN - len(feats), 1))
        feats = np.vstack([pad, feats])
    else:
        feats = feats[-SEQ_LEN:]
    sequences.append(feats)
    targets.append(target)

X_all = np.array(sequences, dtype=np.float32)
y_all = np.array(targets, dtype=np.float32)

# 80/20 split
split_idx = int(len(X_all) * 0.8)
X_train_raw, X_test_raw = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]

# Scale
scaler = StandardScaler()
X_flat = X_train_raw.reshape(-1, len(ALL_COLS))
scaler.fit(X_flat)
X_train = scaler.transform(X_train_raw.reshape(-1, len(ALL_COLS))).reshape(X_train_raw.shape)
X_test = scaler.transform(X_test_raw.reshape(-1, len(ALL_COLS))).reshape(X_test_raw.shape)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Shape: {X_train.shape}")

Snapshots: 152,844 / 306,620 (cutoff=0.5)


Train: 5,208 | Test: 1,302 | Shape: (5208, 50, 33)


## 2. Train ContextConditionedTCN

In [3]:
model = ContextConditionedTCN(raw_size=len(RAW_COLS), context_size=len(CROSS_CANDLE_COLS))
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ContextConditionedTCN: {params:,} parameters")

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)
loss_fn = nn.BCEWithLogitsLoss()

X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_v = torch.tensor(X_test, dtype=torch.float32)
y_v = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)

best_val_loss, best_state, no_improve = float("inf"), None, 0
for epoch in range(1, 51):
    model.train()
    for bx, by in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    model.eval()
    with torch.no_grad():
        vl = loss_fn(model(X_v), y_v).item()
        va = ((torch.sigmoid(model(X_v)) >= 0.5).float() == y_v).float().mean().item()
    marker = ""
    if vl < best_val_loss:
        best_val_loss = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
        marker = " *"
    else:
        no_improve += 1
    if epoch % 5 == 0 or marker:
        print(f"  Epoch {epoch:2d}: val_loss={vl:.4f} val_acc={va * 100:.1f}%{marker}")
    scheduler.step()
    if no_improve >= 8:
        print(f"  Early stop at epoch {epoch}")
        break

if best_state:
    model.load_state_dict(best_state)
print(f"\nBest val_loss: {best_val_loss:.4f}")

ContextConditionedTCN: 94,081 parameters


  Epoch  1: val_loss=0.5874 val_acc=69.5% *


  Epoch  2: val_loss=0.5733 val_acc=71.7% *


  Epoch  3: val_loss=0.5636 val_acc=70.2% *


  Epoch  5: val_loss=0.5702 val_acc=71.2%


  Epoch 10: val_loss=0.6013 val_acc=68.4%


  Early stop at epoch 11

Best val_loss: 0.5636


## 3. Calibrate & Evaluate

In [4]:
model.eval()
with torch.no_grad():
    logits = model(X_v)
    probs_raw = torch.sigmoid(logits).numpy().flatten()

# Isotonic calibration
calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
calibrator.fit(probs_raw, y_test)
probs_cal = calibrator.predict(probs_raw)

acc = accuracy_score(y_test, (probs_cal >= 0.5).astype(int))
brier = brier_score_loss(y_test, probs_cal)
f1 = f1_score(y_test, (probs_cal >= 0.5).astype(int), zero_division=0)

print(
    f"Raw:        Brier={brier_score_loss(y_test, probs_raw):.4f}, prob range=[{probs_raw.min():.3f}, {probs_raw.max():.3f}]"
)
print(f"Calibrated: Brier={brier:.4f}, prob range=[{probs_cal.min():.3f}, {probs_cal.max():.3f}]")
print(f"Acc={acc * 100:.1f}%, F1={f1 * 100:.1f}%")

Raw:        Brier=0.1910, prob range=[0.080, 0.941]
Calibrated: Brier=0.1861, prob range=[0.000, 1.000]
Acc=71.4%, F1=70.6%


## 4. Export Artifacts

In [5]:
model_path = os.path.join(MODELS_DIR, "dnn_v1.pt")
torch.save(model, model_path)
print(f"Model: {model_path} ({os.path.getsize(model_path) / 1024:.0f} KB)")

scaler_path = os.path.join(MODELS_DIR, "dnn_scaler_v1.joblib")
joblib.dump(scaler, scaler_path)
print(f"Scaler: {scaler_path}")

calibrator_path = os.path.join(MODELS_DIR, "dnn_calibrator_v1.joblib")
joblib.dump(calibrator, calibrator_path)
print(f"Calibrator: {calibrator_path}")

feature_cols_path = os.path.join(MODELS_DIR, "dnn_feature_cols_v1.joblib")
joblib.dump(ALL_COLS, feature_cols_path)
print(f"Feature cols: {feature_cols_path} ({len(ALL_COLS)} features)")

feature_config = {
    "model": "dnn_v2",
    "features": ALL_COLS,
    "n_features": len(ALL_COLS),
    "raw_features": len(RAW_COLS),
    "cross_candle_features": len(CROSS_CANDLE_COLS),
    "selection_method": "raw_inputs + cross_candle_indicators",
    "architecture": "ContextConditionedTCN",
    "temporal": True,
    "calibrated": True,
    "parameters": params,
    "elapsed_cutoff": ELAPSED_CUTOFF,
    "seq_len": SEQ_LEN,
    "metrics": {"accuracy": round(acc, 4), "brier": round(brier, 4), "f1": round(f1, 4)},
}
features_json_path = os.path.join(DATA_DIR, "optimal_features_dnn.json")
with open(features_json_path, "w") as f:
    json.dump(feature_config, f, indent=2)
print(f"Config: {features_json_path}")

Model: ../../models/dnn_v1.pt (397 KB)
Scaler: ../../models/dnn_scaler_v1.joblib
Calibrator: ../../models/dnn_calibrator_v1.joblib
Feature cols: ../../models/dnn_feature_cols_v1.joblib (33 features)
Config: ../../data/optimal_features_dnn.json


## 5. Verify with DnnPredictor

In [6]:
from polybot.adapters.dnn_predictor import DnnPredictor
from polybot.ports.predictor import Predictor

predictor = DnnPredictor(
    model_path=model_path,
    feature_cols_path=feature_cols_path,
    scaler_path=scaler_path,
    calibrator_path=calibrator_path,
    temporal=True,
    seq_len=SEQ_LEN,
)

assert isinstance(predictor, Predictor)

# Test with a sample row containing all 33 features
sample_row = {col: float(X_test_raw[0, -1, i]) for i, col in enumerate(ALL_COLS)}
sample_row["candle_id"] = "test_candle"
p = predictor.predict(sample_row)
assert 0.0 <= p <= 1.0

print(f"DnnPredictor v2 loaded! temporal=True, {len(ALL_COLS)} features")
print(f"Sample prediction: P(UP) = {p:.4f}")

DnnPredictor v2 loaded! temporal=True, 33 features
Sample prediction: P(UP) = 0.8125
